## 4.6 卷积层（Convolution Layer） - 多卷积核卷积计算


#### 1. 多卷积核是什么意思

##### 1.1 基本概念
前一节我们已经学习了：
* 多通道输入时，一个卷积核会同时处理所有输入通道
* 一个卷积核在整张图上滑动后，最终只能得到 一张特征图

但真实的 CNN 几乎不会只使用 1 个卷积核。

因为一张图像中往往包含很多不同类型的特征，例如：
* 边缘
* 纹理
* 角点
* 弯曲
* 局部形状

所以在一个卷积层中，通常会同时使用多个卷积核，让它们并行地去提取不同的特征。

这就叫：

>多卷积核卷积计算

##### 1.2 为什么需要多个卷积核
因为一个卷积核本质上只是一个“特征检测器”，它更偏向检测某一种模式。

例如：
* 一个卷积核可能更擅长检测竖直边缘
* 一个卷积核可能更擅长检测水平边缘
* 一个卷积核可能更擅长检测拐角
* 一个卷积核可能更擅长检测某种纹理

如果只有一个卷积核，那么它看到图像的角度太单一了。

而多个卷积核一起工作，才能从不同角度理解同一张图像。 👀

#### 2. 多卷积核的计算本质

##### 2.1 每个卷积核独立工作
在同一个卷积层中，如果有多个卷积核，那么它们之间的计算是相互独立的。

也就是说：
* 第 1 个卷积核扫描整张输入，得到第 1 张特征图
* 第 2 个卷积核扫描整张输入，得到第 2 张特征图
* 第 3 个卷积核扫描整张输入，得到第 3 张特征图
* ……

每个卷积核都有自己单独的一组参数，不共享彼此的权重。

##### 2.2 所有卷积核看的是同一个输入
虽然多个卷积核彼此独立，但它们处理的输入是同一个。

例如输入是一张：

`28 × 28 × 3`

的彩色图像，那么这一层中的每一个卷积核都会对这张 `28 × 28 × 3` 的图像做卷积。

区别只是：
* 不同卷积核参数不同
* 所以提取出的特征也不同

##### 2.3 最后把结果堆叠起来
每个卷积核各自产生一张特征图，

最后把这些特征图在“通道维度”上堆叠起来，

就形成这一层最终的输出。

所以：

>多个卷积核 → 多张特征图 → 组成多通道输出

#### 3. 结合单卷积核来回顾

##### 3.1 一个卷积核的情况
假设输入是：

`28 × 28 × 3`

使用一个卷积核：

`3 × 3 × 3`

如果：
* stride = 1
* padding = 0

那么这个卷积核扫描完整张图后，会得到：

`26 × 26 × 1`

这里的 1 就表示：只有 1 张特征图。

##### 3.2 多个卷积核的情况
如果不是 1 个卷积核，而是 8 个卷积核：
* 每个卷积核都是 3 × 3 × 3
* 每个卷积核都独立扫描整个输入

那么最终输出就会变成：

`26 × 26 × 8`

也就是说：
* 空间尺寸还是 26 × 26
* 但输出通道数变成了 8

##### 3.3 这里最关键的结论
>输出特征图通道数 = 卷积核个数

这是多卷积核卷积计算中最重要的结论之一。

#### 4. 多卷积核计算过程

##### 4.1 第一步：准备输入
假设输入是一张彩色图像：

`5 × 5 × 3`

表示：
* 高度 = 5
* 宽度 = 5
* 通道数 = 3

##### 4.2 第二步：准备多个卷积核
假设这一层有 2 个卷积核，并且每个卷积核大小都是：

`3 × 3 × 3`

注意这里要特别区分两个概念：
* 卷积核大小：3 × 3
* 卷积核深度：必须等于输入通道数 3

所以每个卷积核实际上都是一个三维小块。

##### 4.3 第三步：第一个卷积核先算
第一个卷积核会：
* 同时处理输入的 3 个通道
* 在每个位置做多通道卷积
* 在整张图上滑动
* 最终得到 1 张输出特征图

如果 stride = 1，padding = 0，输出尺寸就是：

`3 × 3`

所以第一个卷积核输出：

`3 × 3 × 1`

##### 4.4 第四步：第二个卷积核再算
第二个卷积核也会对同一个输入重复同样的过程：
* 处理同一张 5 × 5 × 3 输入
* 用自己的参数去做卷积
* 最终得到另一张 3 × 3 的特征图

所以第二个卷积核输出：

`3 × 3 × 1`

##### 4.5 第五步：把两张特征图堆叠
最后把这两张 3 × 3 的特征图沿通道维拼在一起：

就得到最终输出：

`3 × 3 × 2`

这说明：
* 高度 = 3
* 宽度 = 3
* 通道数 = 2

这个 2 就来自于：

>卷积核有 2 个

#### 5. RGB 多通道多卷积核卷积计算示例

##### 5.1 输入与卷积核
假设输入是一张 RGB 图像：

`28 × 28 × 3`

现在这一层设置：
* 卷积核大小：3 × 3
* 卷积核个数：16
* stride = 1
* padding = 1

##### 5.2 每个卷积核分别在做什么
这 16 个卷积核都会做同一件事：
* 同时读取 RGB 三个通道
* 在图像上逐位置滑动
* 计算当前位置的响应值
* 最终形成一张特征图

所以：
* 第 1 个卷积核得到第 1 张特征图
* 第 2 个卷积核得到第 2 张特征图
* ……
* 第 16 个卷积核得到第 16 张特征图

##### 5.3 最终输出形状
因为：
* 输入空间尺寸是 28 × 28
* kernel_size = 3
* padding = 1
* stride = 1

所以每个卷积核输出空间尺寸仍然是：

`28 × 28`

而卷积核一共有 16 个，

所以最终输出就是：

`28 × 28 × 16`

##### 5.4 这个输出表示什么
这说明经过这一层之后，原来的图像已经被转换成了：

16 张不同的特征响应图

每一张都表示一种不同特征在图像中各个位置上的响应程度。 📌

##### 5.6 第二层卷积
根据上一层得到的特征图

`28 × 28 × 16`

为16通道，本层的卷积核也必须要是16通道

比如本层继续设置：
* kernal_size = 3 * 3 * 16
* padding = 1
* stride = 1

所以每个卷积核输出空间尺寸仍然是：

`28 × 28`

##### 5.6 这说明了什么
说明 CNN 在不断做两件事：
* 空间上提取特征
* 通道上扩展特征种类

所以网络越往后，通常会看到：
* 空间尺寸逐渐减小

#### 6. 为什么多个卷积核能学到不同特征

##### 6.1 因为参数不同
多个卷积核虽然结构相同，例如都可能是 3 × 3 × 3，

但它们的参数值不同。

也就是说：
* 核 1 的权重是一组数
* 核 2 的权重是另一组数
* 核 3 的权重又不同

所以它们对同一块输入区域会产生不同响应。

##### 6.2 因为训练过程中会自动分工
在训练开始时，这些卷积核通常是随机初始化的。

随着训练进行，它们会不断更新，慢慢形成“分工”。

例如某些卷积核可能逐渐学会：
* 检测边缘
* 检测颜色变化
* 检测局部纹理
* 检测某种形状片段

所以不是我们手动规定它们各自负责什么，

而是模型在训练中自动学出来的。 🤖

##### 6.3 这就是卷积层的强大之处
多个卷积核一起工作，使得卷积层不再只会看一种模式，

而是能够在同一层中同时提取多种特征。

这也是 CNN 能自动完成特征提取的关键原因之一。

#### 7. 多卷积核与输出通道数的关系

##### 7.1 一个卷积核对应一个输出通道
这个关系必须牢牢记住：
*  个卷积核 → 1 个输出通道
* 8 个卷积核 → 8 个输出通道
* 32 个卷积核 → 32 个输出通道

所以：

`输出通道数 = out_channels = 卷积核个数`

##### 7.2 输入通道数决定什么
输入通道数并不决定输出通道数，

它决定的是：

`每个卷积核的深度`

例如：
* 输入通道数 = 3
    * 那么每个卷积核必须是 k × k × 3
* 输入通道数 = 16
    * 那么每个卷积核必须是 k × k × 16

##### 7.3 两个概念不要混淆
这里非常容易混淆的两个点是：
* 输入通道数：决定每个卷积核有多“厚”
* 卷积核个数：决定最终输出有多少个通道

可以直接记成：
* in_channels 决定 kernel depth
* out_channels 决定 output channels

#### 8. 参数量怎么理解

##### 8.1 一个卷积核的参数量
假设输入通道数为 3，卷积核大小为 3 × 3，

那么一个卷积核参数个数为：

`3 × 3 × 3 = 27`

如果再加 1 个偏置项，那么就是：

`28`

##### 8.2 多个卷积核的参数量
如果这一层有 16 个卷积核，那么总参数量就是：

`(3 × 3 × 3 + 1) × 16 = 448`

这里可以看到：
* 卷积核个数越多
* 输出通道越多
* 参数量也会增加

##### 8.3 但依然比全连接更节省
虽然多卷积核会增加参数量，但由于卷积核本身很小，

整体仍然比全连接层节省得多。

这也是 CNN 在图像任务中高效的重要原因之一。